# M13: Hyperparameter Tuning
We will tune the hyperparameters of our top baseline models: Logistic Regression, LightGBM, and CatBoost.
Since we decided to use SMOTE, we must include SMOTE inside a pipeline so it is only applied to the training folds during cross-validation, preventing data leakage.

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data import load_raw_data
from src.cleaning import clean_data
from src.features import build_features
from src.split import split_data

from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Load data
df = build_features(clean_data(load_raw_data("../data/raw")), scale=False)
X_train, X_test, y_train, y_test = split_data(df)


## RandomizedSearchCV on Top Models

In [2]:
tuned_models = {}

# 1. Logistic Regression
lr_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])
lr_param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l2']
}
lr_search = RandomizedSearchCV(lr_pipeline, lr_param_grid, n_iter=5, scoring='roc_auc', cv=3, random_state=42, n_jobs=-1)
lr_search.fit(X_train, y_train)
tuned_models['LogisticRegression'] = lr_search.best_estimator_
print(f"Logistic Regression Best AUC: {lr_search.best_score_:.4f}")
print(f"Logistic Regression Best Params: {lr_search.best_params_}\n")

# 2. LightGBM
lgb_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', lgb.LGBMClassifier(random_state=42))
])
lgb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 5, 7, -1],
    'model__num_leaves': [15, 31, 63]
}
lgb_search = RandomizedSearchCV(lgb_pipeline, lgb_param_grid, n_iter=10, scoring='roc_auc', cv=3, random_state=42, n_jobs=-1)
lgb_search.fit(X_train, y_train)
tuned_models['LightGBM'] = lgb_search.best_estimator_
print(f"LightGBM Best AUC: {lgb_search.best_score_:.4f}")
print(f"LightGBM Best Params: {lgb_search.best_params_}\n")

# 3. CatBoost
cat_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', CatBoostClassifier(random_state=42, verbose=0))
])
cat_param_grid = {
    'model__iterations': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__depth': [4, 6, 8]
}
cat_search = RandomizedSearchCV(cat_pipeline, cat_param_grid, n_iter=10, scoring='roc_auc', cv=3, random_state=42, n_jobs=-1)
cat_search.fit(X_train, y_train)
tuned_models['CatBoost'] = cat_search.best_estimator_
print(f"CatBoost Best AUC: {cat_search.best_score_:.4f}")
print(f"CatBoost Best Params: {cat_search.best_params_}\n")


D:\ok\Customer-Churn-ML-Project-\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Logistic Regression Best AUC: 0.8284
Logistic Regression Best Params: {'model__penalty': 'l2', 'model__C': 0.1}



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4139, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001275 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 900
[LightGBM] [Info] Number of data points in the train set: 8278, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM Best AUC: 0.8356
LightGBM Best Params: {'model__num_leaves': 15, 'model__n_estimators': 100, 'model__max_depth': 5, 'model__learning_rate': 0.05}



CatBoost Best AUC: 0.8352
CatBoost Best Params: {'model__learning_rate': 0.01, 'model__iterations': 200, 'model__depth': 6}



## Save the Best Model
CatBoost typically performs very well, let's pick the best model from our tuning (based on CV AUC) and save it for future evaluation.

In [3]:
import joblib
import os

best_model_name = max(
    [
        ('LogisticRegression', lr_search.best_score_), 
        ('LightGBM', lgb_search.best_score_), 
        ('CatBoost', cat_search.best_score_)
    ], 
    key=lambda x: x[1]
)[0]

print(f"Overall Best Model: {best_model_name}")

os.makedirs("../models", exist_ok=True)
joblib.dump(tuned_models[best_model_name], "../models/best_churn_model.pkl")
print(f"Saved {best_model_name} pipeline to ../models/best_churn_model.pkl")


Overall Best Model: LightGBM
Saved LightGBM pipeline to ../models/best_churn_model.pkl
